# 82514 · Sesión S22 — Planificar en una rejilla: el almacén, A* y el mapa de coste

**Bloque 7** · jueves 3 de diciembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** El robot del almacén del curso, redondo, se convierte en un punto al inflar los obstáculos con su radio. Sobre esa rejilla se busca el camino del muelle al picking: el frente de onda de Dijkstra, A* con una heurística admisible y con una que se pasa. Después, la rejilla de S19 como mapa, el mapa de coste de Nav2 (la capa de inflado y NavFn) y la ejecución del camino con la ley temporal de S12 y el seguimiento puro, con la odometría de B6, en odom y en map. Con el S22b y el tutorial guiado de Nav2, es la base de la opción B de la entrega 3.

**Se apoya en:** Lynch y Park (2017); Corke (2023); Hart, Nilsson y Raphael (1968); documentación de Nav2 (Jazzy); documentación de numpy y scipy.

**Cómo usarlo en clase.** Sigue el guion de la sesión S22 en las diapositivas del bloque 7. Ejecuta la primera celda una sola vez al empezar; en Colab tarda unos segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez al abrir el cuaderno. En Colab tarda unos segundos: no instala nada.
import sys, time, math, zlib, base64
import numpy as np, scipy, matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.patches import Circle

FIJAS = {'numpy': '2.4.6', 'scipy': '1.16.3', 'matplotlib': '3.10.0'}   # las de Colab con las que se probó el bloque 7
EN_COLAB = 'google.colab' in sys.modules
plt.rcParams['figure.figsize'] = (11, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

for nombre, mod in (('numpy', np), ('scipy', scipy), ('matplotlib', matplotlib)):
    print(nombre, mod.__version__, '' if mod.__version__ == FIJAS[nombre] else f'(probado con {FIJAS[nombre]})')
print('en Colab:', EN_COLAB)
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

> **Aviso para Colab.** Ábrelo al empezar la sesión y ejecuta la primera celda: no instala nada (usa numpy, scipy y matplotlib de Colab) y tarda unos segundos.
>
> **En clase (siete minutos):** las secciones 0, 2 y 4 sin detenerte, y el taller, las secciones 1, 3 y 5. Las cifras tienen que salir como en las diapositivas.
>
> **En casa:** la sección 6 (ejecutar el camino, en odom y en map), la 7 (las métricas de la opción B de la entrega 3) y los ejercicios 1 a 3; las soluciones están al final.
>
> **No se sube:** este cuaderno no tiene tarea en Moodle. La opción B de la entrega 3 sale del cuaderno de S22 (este y el S22b) y del tutorial guiado de Nav2, en Moodle desde el lunes 30 de noviembre.

## 0. El almacén, el robot y la búsqueda (diapositivas 50 y 51)

*En clase, ejecuta esta celda sin detenerte.*

Es el almacén de B6 (S18 y S19): 13 × 9 m en celdas de 10 cm (cada celda, un cuadrado de 10 cm centrado en (columna·0,1; fila·0,1)), dos naves separadas por un muro con una puerta ancha (de x = 0,8 a 2,8 m) y una estrecha (de 9,2 a 10,0 m), y cinco estanterías. Los puntos con nombre: el **muelle** (0,6; 0,6), el **puesto de picking** (8,3; 8,2) y el **puesto de la derecha** (11,5; 6,0). El robot del almacén, un diferencial con LiDAR de la familia del TurtleBot3 (S21), es redondo, de radio **r = 0,28 m**, y va a 0,5 m/s como mucho.

- `libre_para(occ, r)`: las celdas en las que puede estar **el centro** del robot: los obstáculos, inflados con r (el espacio de configuraciones, 𝒞 libre).
- `buscar(coste, ini, meta, peso)`: A* con 8 vecinas. Con `peso = 0` es Dijkstra (h = 0); con 1, la distancia en línea recta, que es admisible; con 2, la heurística doblada.
- `frente_de_onda(coste, meta)` y `bajar(D, ini)`: Dijkstra desde la meta hacia todas partes, y bajar por él desde cualquier celda.
- `largo`, `holgura` y `puerta`: la longitud del camino, su distancia mínima a la pared y por qué puerta cruza el muro. La holgura se mide con `a_la_pared`, hasta la cara de cada celda ocupada, no hasta su centro, que está 5 cm más allá.

In [ ]:
# El almacén y las búsquedas (el mismo código que las diapositivas). Sin detenerse.
import math, heapq
import numpy as np
from scipy.ndimage import distance_transform_edt, binary_erosion

RES, FILAS, COLS = 0.10, 90, 130             # celdas de 10 cm: 13 m en x, 9 m en y


def mapa_almacen():
    """La rejilla de ocupacion del almacen de B6 (True = ocupado). Fila = y/0,1; columna = x/0,1: la celda i esta
    centrada en i·0,1 m (es un cuadrado de 10 cm, de i·0,1 − 0,05 a i·0,1 + 0,05)."""
    occ = np.zeros((FILAS, COLS), bool)
    occ[0, :] = occ[-1, :] = True              # paredes exteriores
    occ[:, 0] = occ[:, -1] = True
    occ[42:47, :] = True                       # el muro central, de y = 4,2 a 4,7 m
    occ[42:47, 8:28] = False                   #   la puerta ancha, de x = 0,8 a 2,8 m
    occ[42:47, 92:100] = False                 #   la puerta estrecha, de x = 9,2 a 10,0 m
    occ[12:36, 40:47] = True                   # estanterias de la nave de abajo
    occ[12:36, 64:71] = True
    occ[58:78, 34:41] = True                   # estanterias de la nave de arriba
    occ[58:78, 58:65] = True
    occ[58:78, 82:89] = True
    return occ


def con_estrecha_cerrada(occ):
    """El mismo almacen con la puerta estrecha cerrada (un palé delante, como en S18)."""
    o = occ.copy()
    o[42:47, 92:100] = True
    return o


OCC = mapa_almacen()
DIST = distance_transform_edt(~OCC) * RES    # distancia de centro a centro de cada celda a la ocupada mas cercana, en m
EXT = (-RES / 2, COLS * RES - RES / 2, -RES / 2, FILAS * RES - RES / 2)   # para imshow: la celda i, centrada en i·0,1 m
MUELLE, PICKING, DERECHA = (0.6, 0.6), (8.3, 8.2), (11.5, 6.0)   # los puntos con nombre del almacen, en m
R_ROBOT, V_MAX, W_MAX = 0.28, 0.5, 1.0       # el robot del almacen: radio, 0,5 m/s y 1 rad/s


def celda(p):
    """De (x; y) en metros a (fila, columna) de la rejilla."""
    return (int(round(p[1] / RES)), int(round(p[0] / RES)))


def libre_para(occ, r):
    """El espacio libre para el CENTRO de un robot redondo de radio r: los obstaculos, inflados con r."""
    return distance_transform_edt(~occ) * RES > r


VECINOS = [(-1, 0, 1.0), (1, 0, 1.0), (0, -1, 1.0), (0, 1, 1.0),
           (-1, -1, math.sqrt(2)), (-1, 1, math.sqrt(2)), (1, -1, math.sqrt(2)), (1, 1, math.sqrt(2))]


def buscar(coste, ini, meta, peso=1.0, devolver_cerrada=False):
    """A* sobre la rejilla, con 8 vecinos. coste[v] es lo que cuesta entrar en v (inf = no se puede).
    peso = 0: Dijkstra (h = 0); 1: A* con la euclidea, admisible; 2: la heuristica doblada, que se pasa.
    Devuelve (camino, coste, nodos expandidos[, conjunto de expandidos])."""
    h = lambda v: peso * math.hypot(v[0] - meta[0], v[1] - meta[1])
    n, m = coste.shape
    g, padre = {ini: 0.0}, {}
    abierta, cerrada = [(h(ini), 0.0, ini)], set()   # OPEN, ordenada por f = g + h; CLOSED
    while abierta:
        _, gu, u = heapq.heappop(abierta)
        if u in cerrada:
            continue
        cerrada.add(u)
        if u == meta:
            break
        for di, dj, paso in VECINOS:
            v = (u[0] + di, u[1] + dj)
            if not (0 <= v[0] < n and 0 <= v[1] < m) or v in cerrada:
                continue
            c = coste[v]
            if not np.isfinite(c):
                continue
            ng = gu + paso * c
            if ng < g.get(v, np.inf):
                g[v] = ng
                padre[v] = u
                heapq.heappush(abierta, (ng + h(v), ng, v))
    if meta not in g:
        res = (None, np.inf, len(cerrada))
    else:
        cam, u = [meta], meta
        while u != ini:
            u = padre[u]
            cam.append(u)
        res = (cam[::-1], g[meta], len(cerrada))
    return res + (cerrada,) if devolver_cerrada else res


def frente_de_onda(coste, meta):
    """Dijkstra desde la meta hacia todas partes: el coste de cada celda hasta la meta (la transformada de distancia)."""
    D = np.full(coste.shape, np.inf)
    D[meta] = 0.0
    monton = [(0.0, meta)]
    while monton:
        d, u = heapq.heappop(monton)
        if d > D[u]:
            continue
        for di, dj, paso in VECINOS:
            v = (u[0] + di, u[1] + dj)
            if not (0 <= v[0] < coste.shape[0] and 0 <= v[1] < coste.shape[1]):
                continue
            if not np.isfinite(coste[v]):
                continue
            nd = d + paso * coste[v]
            if nd < D[v]:
                D[v] = nd
                heapq.heappush(monton, (nd, v))
    return D


def bajar(D, ini, max_pasos=20000):
    """El camino desde cualquier celda: bajar por el frente de onda, al vecino de menor D, hasta la meta (D = 0)."""
    cam, u = [ini], ini
    for _ in range(max_pasos):
        if D[u] == 0.0:
            return cam
        v = min(((u[0] + di, u[1] + dj) for di, dj, _ in VECINOS
                 if 0 <= u[0] + di < D.shape[0] and 0 <= u[1] + dj < D.shape[1]), key=lambda w: D[w])
        if D[v] >= D[u]:
            return None
        u = v
        cam.append(u)
    return None


def largo(cam):
    """Longitud de un camino de celdas, en metros."""
    p = np.array(cam, float) * RES
    return float(np.linalg.norm(np.diff(p, axis=0), axis=1).sum())


_CARAS = np.argwhere(OCC & ~binary_erosion(OCC, border_value=0)) * RES   # (y, x) de las celdas ocupadas del borde


def a_la_pared(xy, caras=None):
    """La distancia (m) de cada punto (x, y) a la cara del obstaculo mas cercano: cada celda ocupada es un cuadrado
    de 10 cm, y se mide hasta su borde, no hasta su centro (que esta 5 cm mas alla)."""
    p = np.atleast_2d(np.asarray(xy, float))
    caras = _CARAS if caras is None else caras
    out = np.empty(len(p))
    for k in range(0, len(p), 500):                              # por tandas, para no gastar memoria
        q = p[k:k + 500]
        dx = np.maximum(np.abs(q[:, None, 0] - caras[None, :, 1]) - RES / 2, 0.0)
        dy = np.maximum(np.abs(q[:, None, 1] - caras[None, :, 0]) - RES / 2, 0.0)
        out[k:k + 500] = np.hypot(dx, dy).min(axis=1)
    return out


def holgura(cam):
    """La distancia minima del centro del robot a la pared (a la cara del obstaculo) a lo largo del camino, en m."""
    c = np.array(cam, float)
    return float(a_la_pared(c[:, ::-1] * RES).min())


def puerta(cam):
    """Por que puerta cruza el muro un camino de celdas."""
    cols = [cc for f, cc in cam if 42 <= f <= 46]
    return "ancha" if cols and max(cols) < 40 else ("estrecha" if cols else "ninguna")


def xy(cam):
    """De una lista de celdas (fila, columna) a (x, y) en metros."""
    c = np.array(cam, float)
    return c[:, 1] * RES, c[:, 0] * RES


def dibuja_almacen(ax, occ=None, libre=None, titulo='', lim=EXT):
    """El almacén: obstáculos en gris oscuro y, si se pasa libre, el C-obstáculo en rosa."""
    occ = OCC if occ is None else occ
    img = np.ones(occ.shape + (3,))
    if libre is not None:
        img[~libre & ~occ] = (0.96, 0.84, 0.82)
    img[occ] = (0.29, 0.31, 0.39)
    ax.imshow(img, origin='lower', extent=EXT, interpolation='nearest')
    ax.set_xlim(lim[0], lim[1]); ax.set_ylim(lim[2], lim[3]); ax.set_aspect('equal')
    ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)
    ax.set_title(titulo)


A, B, C_DER = celda(MUELLE), celda(PICKING), celda(DERECHA)
print(f'el almacén: {OCC.sum()} celdas ocupadas de {OCC.size} ({coma(100 * OCC.mean(), 1)} %)')
fig, ax = plt.subplots(figsize=(6, 4.2))
dibuja_almacen(ax, titulo='El almacén del curso')
for p, nom in ((MUELLE, 'muelle'), (PICKING, 'picking'), (DERECHA, 'puesto de la derecha')):
    ax.plot(*p, 'o', color=IQS_VERDE); ax.annotate(nom, p, (5, 5), textcoords='offset points', color=IQS_AZUL)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
plt.show()

## 1. Taller: inflar con el radio (diapositiva 51)

Inflar los obstáculos con el radio del robot es tratarlo como un punto. Con r = 0,28 m quedan 8100 celdas libres para su centro, el 69,2 %. En la puerta estrecha, de 0,8 m, el centro tiene una franja de 0,8 − 2·0,28 = 0,24 m; en la rejilla quedan 4 columnas por fila (20 celdas), porque la distancia se mide al centro de la celda de la pared, 5 cm detrás de su cara. Las dos columnas de los bordes quedan a 0,25 m de la pared, menos que r: la rejilla da media celda de menos.

**Taller.** Ejecuta, cambia `R_TALLER` a 0,40 m (y después a 0,35) y vuelve a ejecutar. ¿Qué le pasa a la puerta estrecha? ¿Y al camino del muelle al puesto de la derecha?

In [ ]:
# Taller: cambia R_TALLER y ejecuta.
R_TALLER = 0.28      # cámbialo a 0.40 (y prueba 0.35)

fig, axs = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, r in zip(axs, (R_ROBOT, R_TALLER)):
    libre = libre_para(OCC, r)
    est = int(libre[42:47, 92:100].sum())
    cam, cst, nexp = buscar(np.where(libre, 1.0, np.inf), A, C_DER, 1.0)
    print(f'r = {coma(r)} m: {libre.sum()} celdas libres para el centro ({coma(100 * libre.mean(), 1)} %); '
          f'puerta estrecha: franja 0,8 − 2r = {coma(max(0.0, 0.8 - 2 * r))} m, {est} celdas libres; '
          f'muelle -> puesto de la derecha: {coma(cst * RES)} m, por la puerta {puerta(cam)}')
    dibuja_almacen(ax, libre=libre, titulo=f'r = {coma(r)} m: {est} celdas en la estrecha', lim=(8.6, 10.6, 3.6, 5.3))
    f, c = np.where(libre)
    ax.scatter(c * RES, f * RES, s=6, color=IQS_VERDE)
plt.show()

## 2. El frente de onda de Dijkstra (diapositiva 54)

*En clase, ejecuta esta celda sin detenerte.*

Dijkstra lanzado desde la meta expande siempre la celda de menor coste acumulado: g(v) = g(u) + paso·coste(v), con un paso de 1 celda en recto y √2 en diagonal. Al acabar, cada celda libre sabe su coste hasta el picking, y **cualquier otra consulta sale gratis**: se baja por el frente, siempre a la vecina más baja.

In [ ]:
# Una sola búsqueda desde el picking, y tres consultas. Sin detenerse.
LIBRE = libre_para(OCC, R_ROBOT)
BIN = np.where(LIBRE, 1.0, np.inf)             # la rejilla binaria inflada: 1 libre, inf ocupada
t0 = time.time()
D = frente_de_onda(BIN, B)
print(f'el frente de onda desde el picking: {np.isfinite(D).sum()} celdas alcanzadas en {coma((time.time() - t0) * 1000, 0)} ms')
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.imshow(np.where(np.isfinite(D), D * RES, np.nan), origin='lower', extent=EXT, cmap='Blues')
for p, nom in ((MUELLE, 'el muelle'), ((1.0, 8.0), 'la esquina (1,0; 8,0)'), (DERECHA, 'el puesto de la derecha')):
    t0 = time.time()
    cam = bajar(D, celda(p))
    print(f'desde {nom:24s}: {coma(D[celda(p)] * RES)} m, {len(cam)} celdas, en {coma((time.time() - t0) * 1000, 1)} ms')
    ax.plot(*xy(cam), color=AMBAR, lw=2)
ax.set_title('El coste de cada celda hasta el picking (m)'); ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)
plt.show()

## 3. Taller: A* y la heurística (diapositivas 55 y 56)

A* expande la celda de menor f(n) = g(n) + h(n): lo gastado más lo que estima que falta. Si h nunca sobreestima lo que falta (h ≤ h*, **admisible**), A* da el camino óptimo. La distancia en línea recta lo es; la doblada, no.

Primero, la rejilla de 8 × 6 de la diapositiva 55. Después, el almacén.

**Taller.** Ejecuta con los pesos 0 y 1; después añade 2, 1,5 y 3 a `PESOS` y vuelve a ejecutar. Apunta nodos, longitud y puerta. ¿Entre qué dos pesos deja de ser óptimo? El ejercicio 2 lo afina.

In [ ]:
# La rejilla de 8 x 6 de la diapositiva 55: un muro en x = 4, de y = 0 a 3; inicio (1; 1), meta (6; 1).
occ8 = np.zeros((6, 8), bool); occ8[0:4, 4] = True
coste8 = np.where(occ8, np.inf, 1.0)
for w, nom in ((0, 'Dijkstra (h = 0)'), (1, 'A*, con la línea recta'), (2, 'la heurística doblada')):
    cam, cst, nexp = buscar(coste8, (1, 1), (1, 6), w)
    print(f'{nom:24s}: coste {coma(cst, 2)} celdas, {nexp} celdas expandidas')
print('h(inicio) = 5 ≤ h*(inicio) = 5·√2 + 1 =', coma(5 * math.sqrt(2) + 1, 2))

# Taller: el almacén, del muelle al picking, con varios pesos de la heurística.
PESOS = (0, 1)      # añade 2, 1.5 y 3
ref = None
for w in PESOS:
    t0 = time.time()
    cam, cst, nexp = buscar(BIN, A, B, w)
    dt = time.time() - t0
    if ref is None:
        ref = (nexp, cst)
    print(f'peso {coma(w, 1):>4s}: {nexp:5d} nodos ({coma(100 * nexp / ref[0], 1):>5s} %), {coma(cst * RES)} m '
          f'({coma(100 * (cst / ref[1] - 1), 1):>5s} %), puerta {puerta(cam)}, {coma(dt * 1000, 0)} ms')

In [ ]:
# Los nodos expandidos, en azul, con los pesos 0, 1 y 2.
fig, axs = plt.subplots(1, 3, figsize=(12, 3.0))
for ax, w in zip(axs, (0, 1, 2)):
    cam, cst, nexp, cerrada = buscar(BIN, A, B, w, devolver_cerrada=True)
    img = np.ones(OCC.shape + (3,))
    for f, c in cerrada:
        img[f, c] = (0.79, 0.82, 0.93)
    img[OCC] = (0.29, 0.31, 0.39)
    ax.imshow(img, origin='lower', extent=EXT)
    ax.plot(*xy(cam), color=AMBAR, lw=2)
    ax.set_title(f'peso {w}: {nexp} nodos, {coma(cst * RES)} m'); ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 4. La rejilla de S19 como mapa (diapositiva 57)

*En clase, ejecuta esta celda sin detenerte.*

La rejilla de ocupación que hizo el SLAM de S19 (el grafo cerrado y los barridos del LiDAR de S6), guardada aquí en texto para no repetirlo: −1 libre, 0 desconocido, 1 ocupado. Planificando sobre ella, el camino pasa a **0,00 m** del almacén de verdad: cruza celdas que la rejilla da por vistas libres y que de verdad son pared. **El mapa hereda el error de las poses del SLAM**: con hasta 27 cm y 3° de error en las poses del grafo, las paredes salen corridas, y el camino pasa por donde de verdad hay pared. Por eso Nav2 añade una capa con lo que ven los sensores en vivo, y por eso las cifras de este cuaderno salen del almacén de verdad.

In [ ]:
# La rejilla de S19 (zlib + base64) y lo que pasa al planificar sobre ella. Sin detenerse.
REJ19 = '''eNrtWotuwyAM9N3/f/QSHq3aYoqHgaQCTZPW0HCcjR/HKOegMo4HMmQw/LyvLEvGqnU3go3gegi4ObgRB8pEonMbzd+l/gLo
Tz2toM07Pp6FAALAypCrFbSJByx0IKDBCtB3jzl+gFHhgB7mmnYa6Y+ANgQDDIH+szAtLdQnYrwbSPUoLM0L/UeBFg7gj0BM
CAp5gb0nhJbMpAdfTDkM9UM9KyI5b99uhbUhSUPg4YrNxi6WSJjHQWc5eOl+YTkC2RwQXE2B3M0PAuTYrTodzrofBEUzS5uk
Fn8Q+tdPOfQl9it7pVT1zKNNYShmw2OlrH2LlfkvvBKXBySVOkjQcb42/m5MiKh0U2gsg2K1wzg/ra0YIxmb8oVA6ib4VogB
E/oAXjoeyP3iwUbwk5lpn4XfQMAZfqAlz5iROIGDiJKl7QPlz2kRZ9hCdKlboSZq098K5RyatsnORq7JCuWVVAT+HEBBoK1X
lZxcORAXDmQ5gmYOPq2FWBb3xoOm65Fy5EGsdWdwIErk0XbAovI4wBO1M2IMCk3VOmrzuiMS/i8o6hEJ/n5Q2hkqnZ2zH1TY
dpA5OysUdDmhCQEMdwtHK26xQo8srmSggACuCGilewAHMBkXppgId0c0Am7zxFF3C1fpWGQ5giEYcCsr7O79XlZAEH9rA0E1
Bky3JmycjihxBw1antV7jr4htPJxN42PdJKFeaTK8Clva6I2HppxVJ6Z1fC4Zr6IhxKHS/yEFyFfGEjczqmtf2E2gcirARn7
iSX8PxZeKiY8siNSxRgq/YCBmbn0OG7pD9yEBc0='''
s19 = np.frombuffer(zlib.decompress(base64.b64decode(REJ19.replace('\n', ''))), np.int8).reshape(FILAS, COLS) - 1
print(f'la rejilla de S19: {(s19 > 0).sum()} ocupadas, {(s19 < 0).sum()} libres y {(s19 == 0).sum()} desconocidas')
for nom, ocup in (('lo desconocido, libre', s19 > 0), ('lo desconocido, ocupado', s19 >= 0)):
    lib = libre_para(ocup, R_ROBOT)
    cam, cst, nexp = buscar(np.where(lib, 1.0, np.inf), A, B, 1.0)
    print(f'{nom:24s}: {coma(cst * RES)} m; holgura frente al almacén de verdad, {coma(holgura(cam))} m')
cam19, _, _ = buscar(np.where(libre_para(s19 > 0, R_ROBOT), 1.0, np.inf), A, B, 1.0)
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.imshow(s19, origin='lower', extent=EXT, cmap='Greys', vmin=-1, vmax=1.5)
ax.contour(np.arange(COLS) * RES, np.arange(FILAS) * RES, OCC.astype(float), levels=[0.5], colors=[ROJO], linewidths=0.8)
ax.plot(*xy(cam19), color=AMBAR, lw=2)
ax.set_title('La rejilla de S19 y, en rojo, el almacén de verdad'); ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 5. Taller: el mapa de coste de Nav2 (diapositiva 57)

La **capa de inflado** de Nav2 pone a cada celda un coste c(d) que baja con su distancia d al obstáculo más cercano: 254 en el obstáculo (letal), 253 a menos de r (inscrito) y 252·e^(−3·(d − r)) hasta 0,70 m, con los valores por defecto de Jazzy (`cost_scaling_factor` 3,0 e `inflation_radius` 0,70 m). **NavFn**, el planificador global por defecto, paga 50 + 0,8·c por celda y busca con el frente de onda de Dijkstra.

Después se recorre cada camino con el **seguimiento puro** (Corke, 2023), como el controlador del curso en Nav2: a 20 Hz, persigue el punto del camino que está a una anticipación de 0,5 m por delante, con ω = 2·v·sen α / anticipación.

**Taller.** Ejecuta y compara el camino binario con el de NavFn: longitud, holgura al planificarlo y holgura al seguirlo, medidas hasta la cara de la pared. ¿Por qué el más corto no es el mejor?

In [ ]:
# La capa de inflado, NavFn y la ejecución (el mismo código que las diapositivas).
R_INFLADO, K_INFLADO = 0.70, 3.0             # inflation_radius y cost_scaling_factor de Nav2 (valores por defecto)


def capa_inflado(dist, r=R_ROBOT, alcance=R_INFLADO, k=K_INFLADO):
    """El coste de la capa de inflado de Nav2 (0 a 254): 254 en el obstaculo (letal), 253 a menos de r (inscrito),
    252·e^(−k·(d − r)) hasta el alcance y 0 mas alla."""
    c = np.where(dist <= r, 253.0, np.where(dist <= alcance, np.floor(252.0 * np.exp(-k * (dist - r))), 0.0))
    return np.where(dist <= 0, 254.0, c)


def coste_navfn(c):
    """Lo que cuesta pasar por cada celda en NavFn, el planificador global de Nav2: 50 + 0,8·c; inf si c ≥ 253."""
    return np.where(c < 253, 50.0 + 0.8 * c, np.inf)


def trapecio(L, v=V_MAX, a=0.5):
    """La ley temporal de S12, el trapecio: cuanto se tarda en recorrer L metros con v y a maximas."""
    ta = v / a
    return 2 * ta + (L - v * ta) / v if L > v * ta else 2 * math.sqrt(L / a)


def seguimiento_puro(cam, anticipacion=0.5, v=V_MAX, dt=0.05, marco="verdad", semilla=None):
    """El seguimiento puro (Corke, 2023) del camino cam (lista de celdas), con el robot del almacen.
    Cada 50 ms (20 Hz, como el controlador de Nav2) busca en el camino el punto a «anticipacion» metros por delante
    (la zanahoria) y gira hacia el con ω = 2·v·sen α / anticipacion, frenando al final.
    marco = "verdad": el robot sabe donde esta (sin ruido). Con ruido (semilla), el robot real se mueve con la
    odometria de B6 (6 cm y 0,9° tras un metro, que crecen con la raiz de los metros) y el controlador usa su
    estimacion: en "odom", la odometria sola; en "map", la odometria corregida cada segundo por la localizacion (como
    el simulador del curso).
    Devuelve un dict: t (s), desviacion maxima del camino (m), holgura minima a la pared (m), error de llegada (m) y
    la trayectoria verdadera."""
    p = np.array(cam, float)[:, ::-1] * RES                      # (x, y) de cada celda
    x0 = np.array([p[0, 0], p[0, 1], math.atan2(p[3, 1] - p[0, 1], p[3, 0] - p[0, 0])])
    verdad, est = x0.copy(), x0.copy()
    rng = np.random.default_rng(semilla) if semilla is not None else None
    tray, i, t, ult_corr = [verdad[:2].copy()], 0, 0.0, 0.0
    while np.hypot(*(p[-1] - est[:2])) > 0.05 and t < 200:
        d = np.hypot(p[i:, 0] - est[0], p[i:, 1] - est[1])
        i = i + int(d.argmin())
        j = i
        while j < len(p) - 1 and np.hypot(p[j, 0] - est[0], p[j, 1] - est[1]) < anticipacion:
            j += 1
        gx, gy = p[j]
        alfa = math.atan2(gy - est[1], gx - est[0]) - est[2]
        alfa = (alfa + math.pi) % (2 * math.pi) - math.pi
        vv = min(v, 0.5 * np.hypot(*(p[-1] - est[:2])) + 0.05)
        La = max(np.hypot(gx - est[0], gy - est[1]), 0.05) if j == len(p) - 1 else anticipacion
        w = max(-W_MAX, min(W_MAX, 2 * vv * math.sin(alfa) / La))
        ds, dth = vv * dt, w * dt
        est = np.array([est[0] + ds * math.cos(est[2]), est[1] + ds * math.sin(est[2]), est[2] + dth])
        if rng is not None:                                     # el robot real se mueve con el ruido de la odometria
            ds += rng.normal(0, 0.06 * math.sqrt(ds))
            dth += rng.normal(0, math.radians(0.9) * math.sqrt(vv * dt))
        verdad = np.array([verdad[0] + ds * math.cos(verdad[2]), verdad[1] + ds * math.sin(verdad[2]), verdad[2] + dth])
        t += dt
        if rng is None:
            est = verdad.copy()
        elif marco == "map" and t - ult_corr >= 1.0 - 1e-9:     # la localizacion corrige una vez por segundo
            est, ult_corr = verdad.copy(), t
        tray.append(verdad[:2].copy())
    tray = np.array(tray)
    a_, b_ = p[:-1], p[1:]
    ab = b_ - a_

    def dist_pol(q):
        tt = np.clip(((q - a_) * ab).sum(1) / np.maximum((ab * ab).sum(1), 1e-12), 0, 1)
        return np.min(np.hypot(*(a_ + tt[:, None] * ab - q).T))
    desv = max(dist_pol(q) for q in tray)
    return dict(t=t, desviacion=float(desv), holgura=float(a_la_pared(tray).min()),
                llegada=float(np.hypot(*(tray[-1] - p[-1]))), tray=tray)

In [ ]:
# Taller: el camino binario frente al de NavFn.
CM = capa_inflado(DIST)
NAV = coste_navfn(CM)
for d_ in (0.40, 0.50, 0.70):
    print(f'c({coma(d_)}) = {int(math.floor(252 * math.exp(-3 * (d_ - R_ROBOT))))}', end='   ')
print()
CAM_BIN, CST_BIN, _ = buscar(BIN, A, B, 0)
CAM_NAV, _, _ = buscar(NAV, A, B, 0)
PP = {}
for nom, cam in (('binario', CAM_BIN), ('NavFn  ', CAM_NAV)):
    pp = seguimiento_puro(cam, anticipacion=0.5)
    PP[nom] = pp
    print(f'{nom}: {coma(largo(cam))} m, holgura al planificar {coma(holgura(cam))} m; '
          f'al seguirlo, se separa {coma(100 * pp["desviacion"], 1)} cm y la holgura queda en {coma(pp["holgura"])} m '
          f'{"(menos que r: choca)" if pp["holgura"] < R_ROBOT else ""}')
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
axs[0].imshow(np.clip(CM, 0, 254), origin='lower', extent=EXT, cmap='Blues')
axs[0].plot(*xy(CAM_BIN), color=AMBAR, lw=2, label='binario'); axs[0].plot(*xy(CAM_NAV), color=IQS_AZUL, lw=2, label='NavFn')
axs[0].legend(loc='lower right'); axs[0].set_title('El mapa de coste y los dos caminos')
axs[1].imshow(np.clip(CM, 0, 254), origin='lower', extent=EXT, cmap='Blues')
for (nom, pp), col in zip(PP.items(), (AMBAR, IQS_AZUL)):
    axs[1].plot(pp['tray'][:, 0], pp['tray'][:, 1], color=col, lw=2)
axs[1].set_xlim(1.0, 4.2); axs[1].set_ylim(3.3, 6.3); axs[1].set_title('Al seguirlos, en la puerta ancha')
for a in axs:
    a.xaxis.set_major_formatter(COMA); a.yaxis.set_major_formatter(COMA)
plt.show()

## 6. En casa: ejecutar el camino, en odom y en map (diapositivas 50 y 57)

**El tiempo.** Con el trapecio de S12, 0,5 m/s y 0,5 m/s²: t = L/v + v/a.

**El robot de verdad.** Ahora el robot se mueve con el ruido de la odometría de B6 (6 cm y 0,9° tras un metro, que crecen con la raíz de los metros). En **odom**, el controlador solo sabe la odometría, que deriva; en **map**, la localización la corrige una vez por segundo, como el simulador del curso de S21 (y AMCL en Nav2). Veinte semillas para cada caso: el error al llegar y la holgura mínima frente al almacén de verdad.

In [ ]:
# En casa: el tiempo de cada camino y su ejecución con la odometría de B6 (20 semillas; unos segundos).
for nom, cam in (('binario', CAM_BIN), ('NavFn', CAM_NAV)):
    L = largo(cam)
    print(f'{nom:8s}: {coma(L)} m -> {coma(trapecio(L), 1)} s con el trapecio de S12')
print()
for nom, cam in (('binario', CAM_BIN), ('NavFn', CAM_NAV)):
    for marco in ('odom', 'map'):
        R = [seguimiento_puro(cam, 0.5, marco=marco, semilla=s) for s in range(20)]
        ll = np.array([r['llegada'] for r in R]) * 100
        hh = np.array([r['holgura'] for r in R])
        print(f'{nom:8s} en {marco:4s}: llegada {coma(np.median(ll), 1)} cm de mediana (p90 {coma(np.percentile(ll, 90), 1)}); '
              f'holgura mínima {coma(np.median(hh))} m de mediana; choca (holgura < r) en {int((hh < R_ROBOT).sum())} de 20')

## 7. En casa: las métricas de la opción B (diapositiva 62)

La opción B de la entrega 3 compara dos planificadores (o dos costes) con las mismas métricas: **longitud**, **nodos expandidos**, **holgura mínima**, **tiempo de cálculo** y la **ejecución** (el error al llegar en map, la desviación del camino y la holgura al seguirlo). Esta función las calcula para un planificador de rejilla; el cuaderno S22b añade la tasa de éxito en N semillas para los de muestreo.

**Para vuestra entrega 3 (opción B).** El almacén del curso es el ejemplo. En la entrega:
- el mapa es el de vuestro robot: la rejilla de vuestro S19 o el mapa del tutorial guiado de Nav2;
- `R_ROBOT` es su radio: si es un diferencial tipo TurtleBot3, 0,22 m, el que trae el ejemplo de Nav2 (quedan 8156 celdas libres y las mismas 20 en la estrecha); lo hecho aquí vale igual;
- el camino lo ejecuta **el controlador de vuestra entrega 2**, no este seguimiento puro, y se dice con qué localización se mide la llegada;
- se comparan con números dos planificadores, o dos configuraciones de coste, con estas métricas.

Un planificador cuyo camino no ejecuta vuestro controlador pierde la mitad del valor (rúbrica 6.3: el acoplamiento pesa el 30 %). Si vuestro robot es el brazo, el problema es el de la sección 5 del S22b: muestrear en el espacio de sus articulaciones.

In [ ]:
# En casa: la tabla de la opción B para tres planificadores de rejilla.
def metricas(nombre, coste, peso, semillas=10):
    t0 = time.time()
    cam, cst, nexp = buscar(coste, A, B, peso)
    dt = time.time() - t0
    R = [seguimiento_puro(cam, 0.5, marco='map', semilla=s) for s in range(semillas)]
    return dict(planificador=nombre, longitud=largo(cam), nodos=nexp, holgura=holgura(cam), calculo_ms=1000 * dt,
                llegada_cm=100 * np.median([r['llegada'] for r in R]),
                desviacion_cm=100 * np.median([r['desviacion'] for r in R]),
                holgura_al_seguir=np.median([r['holgura'] for r in R]))

tabla = [metricas('Dijkstra, binario', BIN, 0), metricas('A*, binario', BIN, 1), metricas('NavFn, coste de Nav2', NAV, 0)]
print(f'{"planificador":22s} {"longitud":>9s} {"nodos":>6s} {"holgura":>8s} {"cálculo":>8s} {"llegada":>8s} {"desvío":>7s} {"al seguir":>9s}')
for t in tabla:
    print(f'{t["planificador"]:22s} {coma(t["longitud"]):>7s} m {t["nodos"]:6d} {coma(t["holgura"]):>6s} m {coma(t["calculo_ms"], 0):>5s} ms '
          f'{coma(t["llegada_cm"], 1):>5s} cm {coma(t["desviacion_cm"], 1):>4s} cm {coma(t["holgura_al_seguir"]):>7s} m')

### Ejercicio 1 (en casa): la puerta estrecha cerrada y el inflado de Nav2

Con la puerta estrecha cerrada (`con_estrecha_cerrada(OCC)`, el palé de S18), ¿cuánto mide el camino binario del muelle al puesto de la derecha? Después, con el coste de NavFn, compara el inflado por defecto (`alcance=0.70`) con uno de 0,35 m, con las dos puertas abiertas y con la estrecha cerrada. Con Nav2 de verdad lo veréis en el tutorial guiado de Nav2 (en Moodle desde el lunes 30), y con la puerta cerrada en marcha, en S23 (jueves 10).

In [ ]:
# Ejercicio 1: tu código aquí.

### Ejercicio 2 (en casa): ¿desde qué peso deja de ser óptimo?

Recorre los pesos de 1 a 3 en pasos de 0,1 y dibuja los nodos expandidos y la longitud del camino del muelle al picking frente al peso. ¿Desde qué peso sale más largo?

In [ ]:
# Ejercicio 2: tu código aquí.

### Ejercicio 3 (en casa): la anticipación del seguimiento puro

Repite la sección 5 con anticipaciones de 0,3, 0,5 y 1,0 m. ¿Cómo cambian la desviación del camino y la holgura en los dos caminos? ¿Por qué una anticipación larga corta más las esquinas?

In [ ]:
# Ejercicio 3: tu código aquí.

## Soluciones

**Taller, sección 1.** Con r = 0,28 m, 20 celdas en la estrecha, aunque las dos columnas de los bordes quedan a 0,25 m de la pared (media celda de menos). Con r = 0,40 m, la franja es 0,8 − 2·0,40 = 0 y quedan 0 celdas: se cierra. Con 0,35 m, 10 celdas. Muelle → puesto de la derecha: 13,31 m por la estrecha con r = 0,28 m y 14,54 m por la ancha con 0,40 m. Un parámetro continuo cambia la topología.

**Taller, sección 3.** Pesos 0 y 1, y después 2, 1,5 y 3: 6885, 2920, 509, 578 y 275 nodos; 12,66, 12,66, 15,21, 12,66 y 15,29 m. Con 1,5 todavía sale el óptimo aquí, pero sin garantía; con 2, un 20 % más largo, por la estrecha.

**Taller, sección 5.** Binario: 12,66 m; al planificarlo, a 0,21 m de la pared (entre centros de celda serían 0,28: pegado al inflado, con media celda de menos), y al seguirlo, a 0,16 m: menos que r, choca. NavFn: 13,88 m (un 9,6 % más), a 0,45 m al planificarlo y a 0,46 m al seguirlo.

**Sección 6.** En odom, unos 30 y 35 cm al llegar (la deriva de la odometría); en map, unos 5 cm con los dos. El binario choca (holgura menor que r) en las 20 semillas incluso en map; el de NavFn, en ninguna en map (en odom, en 8 de 20).

**Ejercicio 1.**

In [ ]:
# Solución del ejercicio 1
cerr = con_estrecha_cerrada(OCC)
d_cerr = distance_transform_edt(~cerr) * RES
cam, cst, _ = buscar(np.where(d_cerr > R_ROBOT, 1.0, np.inf), A, C_DER, 1)
print(f'binario, la estrecha cerrada: {coma(cst * RES)} m, por la puerta {puerta(cam)}')
for nom, occ_ in (('las dos puertas', OCC), ('la estrecha cerrada', cerr)):
    dd = distance_transform_edt(~occ_) * RES
    for alc in (0.70, 0.35):
        cam, _, _ = buscar(coste_navfn(capa_inflado(dd, alcance=alc)), A, C_DER, 0)
        print(f'NavFn, {nom:20s}, inflado hasta {coma(alc)} m: {coma(largo(cam))} m, por la puerta {puerta(cam)}')

Cerrada la estrecha, el binario va por la ancha: 14,43 m. Con NavFn y el inflado por defecto (0,70 m), 15,03 m por la ancha **aunque la estrecha esté abierta**: el coste de pasar por ella, con 0,24 m de franja, es demasiado alto. Con 0,35 m, 13,96 m por la estrecha; cerrada, 14,48 m por la ancha. Nav2 de verdad (tutorial guiado de Nav2, apartados 3.3 y 3.4), con el mapa de 5 cm, va por la ancha con el inflado por defecto (14,65 m) y por la estrecha con 0,35 m (13,74 m): cifras parecidas, no iguales, porque NavFn interpola el potencial. Y si la puerta se cierra en marcha, Nav2 no recorre esos 14,48 m: replanifica cuando la ve el LiDAR. Lo veréis en S23.

**Ejercicio 2.**

In [ ]:
# Solución del ejercicio 2
pesos = np.round(np.arange(1.0, 3.01, 0.1), 1)
res = [buscar(BIN, A, B, w) for w in pesos]
L = np.array([r[1] * RES for r in res]); N = np.array([r[2] for r in res])
primero = pesos[np.argmax(L > L[0] + 1e-6)]
print('el primer peso que da un camino más largo:', coma(primero, 1))
fig, ax1 = plt.subplots(figsize=(7, 3.4))
ax1.plot(pesos, N, 'o-', color=IQS_AZUL); ax1.set_xlabel('peso de la heurística'); ax1.set_ylabel('nodos expandidos', color=IQS_AZUL)
ax2 = ax1.twinx(); ax2.plot(pesos, L, 's-', color=AMBAR); ax2.set_ylabel('longitud (m)', color=AMBAR)
ax1.xaxis.set_major_formatter(COMA); ax2.yaxis.set_major_formatter(COMA)
plt.show()

Los nodos bajan deprisa con el peso, aunque no siempre (con 1,9, 547). La longitud se queda en 12,66 m hasta el peso 1,6; con 1,7 ya sale algo más largo (12,75 m, por la ancha), y con 2 salta a la estrecha: 15,21 m. Una heurística que se pasa no siempre da un camino peor, pero ya no lo garantiza.

**Ejercicio 3.**

In [ ]:
# Solución del ejercicio 3
for ant in (0.3, 0.5, 1.0):
    for nom, cam in (('binario', CAM_BIN), ('NavFn', CAM_NAV)):
        pp = seguimiento_puro(cam, anticipacion=ant)
        print(f'anticipación {coma(ant, 1)} m, {nom:8s}: se separa {coma(100 * pp["desviacion"], 1)} cm; holgura {coma(pp["holgura"])} m')

Con 0,3, 0,5 y 1,0 m, el binario se separa unos 5, 10 y 23 cm y queda a 0,20, 0,16 y 0,10 m de la pared: siempre por debajo de r. El de NavFn, a 0,46, 0,46 y 0,42 m. Una zanahoria lejana tira del robot por la cuerda de cada curva: corta las esquinas. Por eso el camino tiene que dejar margen, y eso es lo que hace el mapa de coste.

---

## Para llevarse de esta sesión

- **Inflar con el radio es tratar al robot como un punto.** Un parámetro continuo, r, cambia la topología: con 0,40 m se cierra la puerta estrecha.
- **El frente de onda da el coste de cada celda hasta la meta**: cualquier consulta a esa meta sale gratis.
- **A* con una heurística admisible da el óptimo con menos trabajo**; con una que se pasa, menos trabajo aún y sin garantía (509 nodos, +20 %).
- **La rejilla da media celda de menos**: mide entre centros de celda, y la pared está 5 cm más cerca. La holgura se mide a la cara de la pared.
- **El camino más corto no es el mejor**: pegado al inflado, al seguirlo choca. El mapa de coste de Nav2 paga un 9,6 % más de camino por 0,45 m de holgura.
- **Moverse en odom, llegar en map**: la odometría sola llega a unos 30 cm; corregida por la localización, a unos 5.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*